In [1]:
mock_page_payload = {
    "url": "https://example.com/ai_article",
    "title": "How AI is Transforming the World",
    "indexed_blocks": 
    [
    {"id": 1, "text": "[1] <div><p>Advertisement: ...</p></div>"},
    
    {"id": 2, "text": "[2] <h1>AI Helps Humans</h1>"},
    
    {"id": 3, "text": "[3] <p>AI can assist humans in many fields.</p>"},
    
    {"id": 4, "text": "[4] <ul><li>Healthcare: ...</li></ul>"},
    
    {"id": 5, "text": "[5] <li>Education: ...</li>"},
    
    {"id": 6, "text": "[6] <div><h2>Related Articles</h2></div>"},
    
    {"id": 7, "text": "[7] <p>The Future of Robotics</p>"}
    ]
}

In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# load api keys from .env file
try:
    from dotenv import load_dotenv
    load_dotenv(project_root / ".env")
except ImportError:
    pass

from main import predict_relevant_block_ids

query = "What are practical ways AI helps people?"

selected_ids = predict_relevant_block_ids(
    query=query,
    page_payload=mock_page_payload,
    model="qwen-flash",
)

print("query:", query)
print("selected_ids:", selected_ids)


query: What are practical ways AI helps people?
selected_ids: [2, 3, 4, 5]


In [3]:
from main import reconstruct_html_from_block_ids

reconstructed_result = reconstruct_html_from_block_ids(
    selected_ids=selected_ids,
    page_payload=mock_page_payload,
    keep_page_order=True,
    wrap_container=True,
)

# print("reconstructed_title:", reconstructed_result["title"])
# print("reconstructed_url:", reconstructed_result["url"])
print("reconstructed_html:")
print(reconstructed_result["html"])


reconstructed_html:
<!doctype html>
<html>
<head>
<title>How AI is Transforming the World</title>
</head>
<body>
<div class="extracted-evidence">
<h1>AI Helps Humans</h1>
<p>AI can assist humans in many fields.</p>
<ul><li>Healthcare: ...</li></ul>
<li>Education: ...</li>
</div>
</body>
</html>


In [4]:
from main import html_to_markdown

reconstructed_result["markdown"] = html_to_markdown(reconstructed_result["html"])

print("reconstructed_markdown:")
print(reconstructed_result["markdown"])


reconstructed_markdown:
How AI is Transforming the World

# AI Helps Humans

AI can assist humans in many fields.

- Healthcare: ...

- Education: ...


In [5]:
from main import answer_query_from_extract_results

# ExtractResults can contain multiple webpages, not just one.
extract_results = [
    {
        **reconstructed_result,
        "doc_id": "doc_1",
    },
    {
        "doc_id": "doc_2",
        "title": "AI in Robotics",
        "url": "https://example.com/ai-robotics",
        "html": "",
        "selected_ids": [],
        "markdown": "AI also supports robotics in manufacturing.",
    },
    {
        "doc_id": "doc_3",
        "title": "Weekend Hiking Tips",
        "url": "https://example.com/hiking-tips",
        "html": "",
        "selected_ids": [],
        "markdown": "Bring water, wear layered clothing, and start early to avoid midday heat on mountain trails.",
    }
]

qa_result = answer_query_from_extract_results(
    query=query,
    extract_results=extract_results,
    model="qwen-flash",
)

print("final_answer:", qa_result["answer"])
print("documents_used:", qa_result["documents_used"])


final_answer: AI helps people in healthcare, education, and robotics in manufacturing.
documents_used: 3
